# GeoLegal-AI E1 baseline (Google Colab)
This notebook runs Tiny Aya without retrieval. It is a research experiment, not legal advice. Dataset selection and versioned output paths are explicit. Inference is disabled by default and must be deliberately enabled.

In [ ]:
%pip install -q "transformers>=5.15,<6" "accelerate>=1.2,<2" safetensors


In [ ]:
# Store HF_TOKEN in Colab Secrets (key icon). Never paste it into the notebook.
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access.')
login(token=hf_token, add_to_git_credential=False)
del hf_token

In [ ]:
from pathlib import Path
import json, sys, torch

REPO_ROOT = Path('/content/geolegal-ai-lexsi')
if not REPO_ROOT.exists():
    raise FileNotFoundError('Clone the repository to /content/geolegal-ai-lexsi first.')
sys.path.insert(0, str(REPO_ROOT))
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')

from src.dataset import load_scenarios
from src.baseline import load_model_once, generate_one, MODEL_ID, MODEL_REVISION
from src.evaluate import score_predictions, save_metrics

DATASET_REVISION = 'held_out_approved_v1'
DATASET_OPTIONS = {
    'held_out_initial': REPO_ROOT / 'data/eval/held_out_initial.jsonl',
    'held_out_proposed_v2': REPO_ROOT / 'data/eval/held_out_proposed_v2.jsonl',
    'held_out_approved_v1': REPO_ROOT / 'data/eval/held_out_approved_v1.jsonl',
}
if DATASET_REVISION not in DATASET_OPTIONS:
    raise ValueError(f'Unknown dataset revision: {DATASET_REVISION}')
DATASET_PATH = DATASET_OPTIONS[DATASET_REVISION]
PREDICTIONS_PATH = REPO_ROOT / f'results/baseline_predictions_{DATASET_REVISION}.jsonl'
METRICS_PATH = REPO_ROOT / f'results/baseline_metrics_{DATASET_REVISION}.json'
RUN_INFERENCE = False  # Change to True only when intentionally starting a fresh comparable run.
scenarios = load_scenarios(DATASET_PATH)
print(f'Loaded {len(scenarios)} scenarios from {DATASET_PATH}')
print(f'Outputs will use {PREDICTIONS_PATH.name} and {METRICS_PATH.name}')
print(torch.cuda.get_device_name(0))

In [ ]:
# Load the tokenizer and model exactly once, and only after inference is explicitly enabled.
tokenizer = model = torch_module = None
if not RUN_INFERENCE:
    print('Model loading skipped because RUN_INFERENCE=False.')
else:
    tokenizer, model, torch_module = load_model_once(MODEL_ID, MODEL_REVISION)
    print(MODEL_ID, MODEL_REVISION, model.device)

In [ ]:
predictions = []
if not RUN_INFERENCE:
    print('Inference is disabled. Set RUN_INFERENCE=True in the selection cell to run.')
else:
    PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with PREDICTIONS_PATH.open('x', encoding='utf-8') as handle:
        for index, scenario in enumerate(scenarios, 1):
            prediction = generate_one(tokenizer, model, torch_module, scenario)
            predictions.append(prediction)
            handle.write(json.dumps(prediction, ensure_ascii=False) + '\n')
            print(f'{index}/{len(scenarios)} {scenario.scenario_id}')
    print(f'Saved {len(predictions)} outputs to {PREDICTIONS_PATH}')

In [ ]:
# Score only predictions produced in this selected run; never substitute older baseline outputs.
if not RUN_INFERENCE:
    print('Scoring skipped because inference was not run.')
else:
    metrics = score_predictions(scenarios, predictions)
    save_metrics(METRICS_PATH, metrics)
    print(json.dumps(metrics, indent=2))